# TP 3 (Transformer) - LayerNorm et MLP

**Cours NLP et Transformers - CY Tech**

Objectif : implementer une LayerNorm, comprendre son axe de calcul, puis
construire le reseau feed-forward applique separement a chaque token.

Contrairement a l'attention, ces operations ne melangent pas les positions : la
meme transformation agit sur chaque vecteur independamment.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(1337)

## Etape 1 - Coder LayerNorm

Pour chaque vecteur, on calcule la moyenne et la variance de ses composantes,
on normalise, puis on reintroduit deux parametres appris gamma et beta.
L'axe de normalisation est la derniere dimension (les features), pas le batch.

In [2]:
class LayerNormMaison:
    def __init__(self, dimension, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dimension)
        self.beta = torch.zeros(dimension)

    def __call__(self, x):
        moyenne = x.mean(dim=1, keepdim=True)
        variance = x.var(dim=1, keepdim=True, unbiased=False)
        x_normalise = (x - moyenne) / torch.sqrt(variance + self.eps)
        return self.gamma * x_normalise + self.beta

## Etape 2 - Choisir l'axe de normalisation

LayerNorm normalise chaque token a travers ses features ; elle ne force pas une
feature donnee a etre normalisee a travers tout le batch. On verifie donc que
c'est la ligne d'un token (pas une colonne de feature) qui a moyenne proche de 0
et ecart-type proche de 1, puis on valide contre `nn.LayerNorm`.

In [3]:
module = LayerNormMaison(100)
x = torch.randn(32, 100)
sortie = module(x)

print("colonne (feature 0) : moyenne", round(sortie[:, 0].mean().item(), 3),
      "ecart-type", round(sortie[:, 0].std().item(), 3))
print("ligne (token 0)     : moyenne", round(sortie[0, :].mean().item(), 3),
      "ecart-type", round(sortie[0, :].std().item(), 3))

reference = nn.LayerNorm(100)
print("egale nn.LayerNorm :", torch.allclose(module(x), reference(x), atol=1e-6))

colonne (feature 0) : moyenne 0.148 ecart-type 0.885
ligne (token 0)     : moyenne -0.0 ecart-type 1.005
egale nn.LayerNorm : True


## Etape 3 - Construire le reseau feed-forward

Le MLP augmente temporairement la dimension (C vers 4C), applique une
non-linearite, puis revient a la dimension de l'embedding. ReLU ramene a zero
les reponses negatives : environ la moitie des activations restent positives.

In [4]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.reseau = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.reseau(x)


ffwd = FeedForward(64)
x = torch.randn(16, 10, 64)
delta = ffwd(x)
print("forme de la sortie :", delta.shape)

couche_haute = nn.Linear(64, 4 * 64)
activation = torch.relu(couche_haute(x))
print("proportion d'activations positives :", round((activation > 0).float().mean().item(), 3))

forme de la sortie : torch.Size([16, 10, 64])
proportion d'activations positives : 0.5


## Etapes 4 et 5 - Ajouter le residuel et comparer les variantes

Le bloc suit une organisation pre-norm : LayerNorm avant la sous-couche, puis
addition residuelle. On remplace ReLU par GELU (plus lisse, utilisee par GPT),
on teste plusieurs facteurs d'expansion et on compte les parametres.

In [5]:
class FeedForwardGelu(nn.Module):
    def __init__(self, n_embd, facteur=4):
        super().__init__()
        self.reseau = nn.Sequential(
            nn.Linear(n_embd, facteur * n_embd),
            nn.GELU(),
            nn.Linear(facteur * n_embd, n_embd),
        )

    def forward(self, x):
        return self.reseau(x)


n_embd = 64
layer_norm = nn.LayerNorm(n_embd)
ffwd = FeedForwardGelu(n_embd)

x = torch.randn(16, 10, n_embd)
x_sortie = x + ffwd(layer_norm(x))   # pre-norm puis connexion residuelle

print("forme en entree :", x.shape)
print("forme en sortie :", x_sortie.shape)

forme en entree : torch.Size([16, 10, 64])
forme en sortie : torch.Size([16, 10, 64])


In [6]:
def compter_parametres(module):
    return sum(p.numel() for p in module.parameters())

for facteur in [2, 4, 8]:
    reseau = FeedForwardGelu(n_embd, facteur)
    print("facteur", facteur, ":", compter_parametres(reseau), "parametres")

facteur 2 : 16576 parametres
facteur 4 : 33088 parametres
facteur 8 : 66112 parametres


## Bilan

- LayerNorm conserve la dimension : elle remet les composantes d'un vecteur sur
  une echelle comparable, sans melanger les tokens ni ajouter de connaissance.
- Le MLP ne melange pas les positions : la meme transformation agit sur chaque
  token. L'attention, elle, fait circuler l'information entre les tokens.
- La connexion residuelle ajoute la correction au vecteur initial au lieu de le
  remplacer, ce qui stabilise l'apprentissage en profondeur.
- Le nombre de parametres croit avec le facteur d'expansion.